# Transformer Zero: Colab run
Set a GPU runtime and a `HF_TOKEN` secret (Notebook access on), then run top to bottom. Training uploads the best adapter and logs to the HF repo after every epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
!pip -q install -U transformers peft accelerate scikit-learn 2>&1 | tail -2
!pip -q uninstall -y torchao  # old Colab torchao breaks the new peft import

In [ ]:
import os, subprocess, re, time
from huggingface_hub import snapshot_download, upload_folder
REPO = 'wajih-rathore/transformer-zero'
snapshot_download(REPO, allow_patterns=['code/*'], local_dir='/content/tz')
os.chdir('/content/tz/code')
def push(folder, path_in_repo):
    upload_folder(repo_id=REPO, folder_path=folder, path_in_repo=path_in_repo, commit_message=f'colab: {path_in_repo}')
def run(cmd, log=None):
    print('$', cmd, flush=True)
    if log:
        return subprocess.Popen(f'{cmd} > {log} 2>&1', shell=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print('
'.join(l for l in (r.stdout + r.stderr).splitlines() if 'arn' not in l and '%|' not in l)[-3000:])
    r.check_returncode()
def status(log, proc=None):
    s = open(log).read()
    bars = re.findall(r'(\d+)/(\d+) \[([\d:]+)<([\d:?]+)', s)
    print('progress:', bars[-1] if bars else 'starting')
    for l in re.split(r'[
]', s):
        if l.startswith(('VAL', 'DONE', 'Traceback')) or "'loss'" in l or 'Error' in l: print(l[:300])
    if proc is not None: print('running' if proc.poll() is None else f'exit code {proc.returncode}')

In [ ]:
run('python scripts/benchmark.py --target Qwen/Qwen3-0.6B --draft Qwen/Qwen3-0.6B --parts correctness --repeats 1 --out results/smoke_0.6b.json')

In [ ]:
M4 = 'Qwen/Qwen3-4B-Instruct-2507'
run(f'python scripts/evaluate.py --model {M4} --name qwen3-4b-base --splits val test')
push('results', 'results')

In [ ]:
train4 = run(f'python scripts/train.py --model {M4} --out runs/qwen3-4b --push {REPO}:runs/qwen3-4b', log='/content/train_4b.log')

In [ ]:
status('/content/train_4b.log', train4)

In [ ]:
run(f'python scripts/evaluate.py --model {M4} --adapter runs/qwen3-4b/best --name qwen3-4b-lora --splits val test')
push('results', 'results')

In [ ]:
run(f'python scripts/benchmark.py --target {M4} --draft Qwen/Qwen3-0.6B --adapter runs/qwen3-4b/best --out results/bench_qwen3-4b.json')
push('results', 'results')

In [ ]:
M17 = 'Qwen/Qwen3-1.7B'
run(f'python scripts/evaluate.py --model {M17} --name qwen3-1.7b-base --splits val test')
run(f'python scripts/train.py --model {M17} --out runs/qwen3-1.7b --push {REPO}:runs/qwen3-1.7b')
run(f'python scripts/evaluate.py --model {M17} --adapter runs/qwen3-1.7b/best --name qwen3-1.7b-lora --splits val test')
push('results', 'results')